# Convert balanced h5ad to parquet and split

In [1]:
import os
from deeptan.utils.data import (
    read_h5ad,
    h5ad_to_parquet,
    JointStratifiedSplitter,
)

/home/wuch/miniforge3/envs/pt28/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/wuch/miniforge3/envs/pt28/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/wuch/.local/lib/python3.12/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSsb
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/wuch/miniforge3/envs/pt28/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /home/wuch/.local/lib/python3.12/site-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSsb
  warnings.warn

## Start

In [4]:
seeds = [i + 42 for i in range(5)]

### GSE226097 balanced

In [5]:
path_h5ad = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_balanced_flower_seedling_rosette.h5ad"

In [6]:
path_parquet = path_h5ad.replace(".h5ad", ".parquet")
if not os.path.exists(path_parquet):
    h5ad_to_parquet(path_h5ad, path_parquet, True)

Read /mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_balanced_flower_seedling_rosette.h5ad with 12500 cells and 19020 features.
Saving to /mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_balanced_flower_seedling_rosette.parquet...
X shape: (12500, 19020)
DataFrame shape: (12500, 19021)
Head of DataFrame:
shape: (5, 19_021)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01010 ┆ AT1G01020 ┆ AT1G01030 ┆ … ┆ AT5G44585 ┆ AT5G53950 ┆ AT5G54148 ┆ AT5G6362 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ 5        │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ GTGTAACGT ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.

In [7]:
celltypes = read_h5ad(path_h5ad).obs["CellType"].to_list()
batch_id = read_h5ad(path_h5ad).obs["batch"].to_list()

print(f"Number of cell types: {len(set(celltypes))}")
print(f"{set(celltypes)}")
print(f"Number of batches: {len(set(batch_id))}")
print(f"{set(batch_id)}")

Number of cell types: 5
{'Epidermal', 'Meristematic', 'Mesophyll', 'Stele', 'Guard'}
Number of batches: 34
{'8-seedling', '19-seedling', '15-seedling', '5-seedling', '13-seedling', '6-seedling', '3-flower', '6-flower', '20-seedling', '1-flower', '2-flower', '4-flower', '9-seedling', '4-rosette', '2-rosette', '5-flower', '22-seedling', '6-rosette', '1-rosette', '18-seedling', '2-seedling', '12-seedling', '4-seedling', '21-seedling', '7-seedling', '11-seedling', '3-rosette', '17-seedling', '14-seedling', '1-seedling', '10-seedling', '16-seedling', '5-rosette', '3-seedling'}


In [8]:
def _tmp_run(_ratio, _seeds, _suff: str):
    output_dir = path_parquet.replace(".parquet", "") + "_split" + "_" + _suff

    _splitter = JointStratifiedSplitter(
        cell_types=celltypes,
        orig_idents=batch_id,
        parquet_file=path_parquet,
        output_dir=output_dir,
        ratio=_ratio,
        seeds=_seeds,
        balance_strategy="none",
    )

    _splitter.execute()                                                                                                                                                                             

In [9]:
_tmp_run([0.8] + [0.1, 0.1], seeds, "full")
_tmp_run([0.4, 0.4] + [0.1, 0.1], seeds, "half")
_tmp_run([0.2, 0.2, 0.2, 0.2] + [0.1, 0.1], seeds, "quarter")
_tmp_run([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1] + [0.1, 0.1], seeds, "eighth")

Created split 0 (seed 42) with 10005 samples
Created split 1 (seed 42) with 1254 samples
Created split 2 (seed 42) with 1241 samples
Created split 0 (seed 43) with 10005 samples
Created split 1 (seed 43) with 1254 samples
Created split 2 (seed 43) with 1241 samples
Created split 0 (seed 44) with 10005 samples
Created split 1 (seed 44) with 1254 samples
Created split 2 (seed 44) with 1241 samples
Created split 0 (seed 45) with 10005 samples
Created split 1 (seed 45) with 1254 samples
Created split 2 (seed 45) with 1241 samples
Created split 0 (seed 46) with 10005 samples
Created split 1 (seed 46) with 1254 samples
Created split 2 (seed 46) with 1241 samples
Created split 0 (seed 42) with 4998 samples
Created split 1 (seed 42) with 4998 samples
Created split 2 (seed 42) with 1254 samples
Created split 3 (seed 42) with 1250 samples
Created split 0 (seed 43) with 4998 samples
Created split 1 (seed 43) with 4998 samples
Created split 2 (seed 43) with 1254 samples
Created split 3 (seed 43) w

### 独立测试

In [3]:
path_h5ad = "./ath_snrna_indeptest_flower_seedling_rosette.h5ad"
path_parquet = path_h5ad.replace(".h5ad", ".parquet")
if not os.path.exists(path_parquet):
    h5ad_to_parquet(path_h5ad, path_parquet, True)

Read ./ath_snrna_indeptest_flower_seedling_rosette.h5ad with 27927 cells and 19020 features.
Saving to ./ath_snrna_indeptest_flower_seedling_rosette.parquet...
X shape: (27927, 19020)
DataFrame shape: (27927, 19021)
Head of DataFrame:
shape: (5, 19_021)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01010 ┆ AT1G01020 ┆ AT1G01030 ┆ … ┆ AT5G44585 ┆ AT5G53950 ┆ AT5G54148 ┆ AT5G6362 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ 5        │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AAAGGTATC ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ TAGTACG-1 ┆           ┆           ┆  

In [6]:
import polars as pl

In [13]:
path_extra = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_balanced_flower_seedling_rosette_split_full_extra"

In [14]:
if not os.path.exists(path_extra):
    os.makedirs(path_extra)

In [10]:
path_extra_test = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_indeptest_flower_seedling_rosette.parquet"
df_extra_test = pl.read_parquet(path_extra_test)
print(df_extra_test)

shape: (27_927, 19_021)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01010 ┆ AT1G01020 ┆ AT1G01030 ┆ … ┆ AT5G44585 ┆ AT5G53950 ┆ AT5G54148 ┆ AT5G6362 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ 5        │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AAAGGTATC ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ TAGTACG-1 ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ _1-flower ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ AACAAGAGT ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0

In [16]:
seeds = [42, 43, 44, 45, 46]

In [17]:
for _s in seeds:
    _path = os.path.join(path_extra, f"split_{_s}_2.parquet")
    df_test = pl.read_parquet(_path)
    print(df_test.shape)
    df_test = pl.concat([df_test, df_extra_test])
    print(df_test.shape, "\n")
    df_test.write_parquet(_path)

(1241, 19021)
(29168, 19021) 

(1241, 19021)
(29168, 19021) 

(1241, 19021)
(29168, 19021) 

(1241, 19021)
(29168, 19021) 

(1241, 19021)
(29168, 19021) 

